In [15]:
import pandas as pd
import unicodedata
import re
from difflib import SequenceMatcher
from itertools import combinations
from pathlib import Path
import numpy as np

In [14]:
#Visualización de las primeras filas 

DATA_PATH = Path('/Users/valemoravale/Documents/UNIVERSIDAD /Semestre 5/ETL/workshop3/data/2018.csv')
df = pd.read_csv(DATA_PATH)
df.head()

,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.632,1.305,1.592,0.874,0.681,0.202,0.393
1,2,Norway,7.594,1.456,1.582,0.861,0.686,0.286,0.340
2,3,Denmark,7.555,1.351,1.590,0.868,0.683,0.284,0.408
3,4,Iceland,7.495,1.343,1.644,0.914,0.677,0.353,0.138
4,5,Switzerland,7.487,1.420,1.549,0.927,0.660,0.256,0.357


In [16]:
#Numero de filas, columnas y tipo de cada una
print('Rows:', len(df))
print('Columns:', df.shape[1])
df.info()

Rows: 156
Columns: 9
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 156 entries, 0 to 155
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Overall rank                  156 non-null    int64  
 1   Country or region             156 non-null    object 
 2   Score                         156 non-null    float64
 3   GDP per capita                156 non-null    float64
 4   Social support                156 non-null    float64
 5   Healthy life expectancy       156 non-null    float64
 6   Freedom to make life choices  156 non-null    float64
 7   Generosity                    156 non-null    float64
 8   Perceptions of corruption     155 non-null    float64
dtypes: float64(7), int64(1), object(1)
memory usage: 11.1+ KB


In [17]:
#Valores faltantes y su porcentaje 
missing = df.isna().sum().to_frame('missing_count')
missing['missing_percent'] = (missing['missing_count'] / len(df) * 100).round(2)
missing

,missing_count,missing_percent
Overall rank,0,0.00
Country or region,0,0.00
Score,0,0.00
GDP per capita,0,0.00
Social support,0,0.00
Healthy life expectancy,0,0.00
Freedom to make life choices,0,0.00
Generosity,0,0.00
Perceptions of corruption,1,0.64


In [18]:
#Columnas duplicadas y Country duplicado 
print('Full duplicated rows:', df.duplicated().sum())
print('Duplicated Country or region:', df['Country or region'].duplicated().sum())

Full duplicated rows: 0
Duplicated Country or region: 0


In [19]:
# Crear función para limpiar nombres
def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Crear columna limpia
df["Country or region_clean"] = df["Country or region"].apply(limpiar_texto)

# Buscar duplicados exactos después de limpiar
duplicados_limpios = df[df.duplicated("Country or region_clean", keep=False)]

print("Duplicados después de limpiar:")
print(duplicados_limpios[["Country or region", "Country or region_clean"]])

Duplicados después de limpiar:
Empty DataFrame
Columns: [Country or region, Country or region_clean]
Index: []


In [20]:
Country_or_region = df["Country or region"].dropna().unique()

posibles_duplicados = []

for pais1, pais2 in combinations(Country_or_region, 2):
    pais1_clean = limpiar_texto(pais1)
    pais2_clean = limpiar_texto(pais2)

    similitud = SequenceMatcher(None, pais1_clean, pais2_clean).ratio()

    if similitud >= 0.85:
        posibles_duplicados.append({
            "Country or region 1": pais1,
            "Country or region 2": pais2,
            "Similitud": round(similitud, 3)
        })

posibles_duplicados_df = pd.DataFrame(posibles_duplicados)

print(posibles_duplicados_df)

  Country or region 1 Country or region 2  Similitud
0             Iceland             Ireland      0.857
1           Australia             Austria      0.875


In [21]:
# Seleccionar solo columnas numéricas
numeric_cols = df.select_dtypes(include=["number"]).columns

ceros = (df[numeric_cols] == 0).sum()
negativos = (df[numeric_cols] < 0).sum()

print("Ceros por columna:")
print(ceros)
print("-------------------------------------")
print("Negativos por columna:")
print(negativos)

Ceros por columna:
Overall rank                    0
Score                           0
GDP per capita                  1
Social support                  1
Healthy life expectancy         1
Freedom to make life choices    1
Generosity                      1
Perceptions of corruption       2
dtype: int64
-------------------------------------
Negativos por columna:
Overall rank                    0
Score                           0
GDP per capita                  0
Social support                  0
Healthy life expectancy         0
Freedom to make life choices    0
Generosity                      0
Perceptions of corruption       0
dtype: int64


In [22]:
filas_no_validas = df[df[numeric_cols].le(0).any(axis=1)]

print(filas_no_validas)

     Overall rank         Country or region  Score  GDP per capita  \
66             67                   Moldova  5.640           0.657   
78             79                    Greece  5.358           1.154   
92             93    Bosnia and Herzegovina  5.129           0.915   
97             98                   Somalia  4.982           0.000   
112           113              Sierra Leone  4.571           0.256   
141           142                    Angola  3.795           0.730   
154           155  Central African Republic  3.083           0.024   

     Social support  Healthy life expectancy  Freedom to make life choices  \
66            1.301                    0.620                         0.232   
78            1.202                    0.879                         0.131   
92            1.078                    0.758                         0.280   
97            0.712                    0.115                         0.674   
112           0.813                    0.000     

In [10]:
# Lista de regiones de referencia
regiones_referencia = [
    "western europe",
    "north america",
    "australia and new zealand",
    "middle east and northern africa",
    "latin america and caribbean",
    "southeastern asia",
    "central and eastern europe",
    "eastern asia",
    "subsaharan africa",
    "southern asia"
]

def limpiar_texto(texto):
    texto = str(texto).strip().lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.replace("&", "and")
    texto = re.sub(r"[^a-z0-9\s]", "", texto)
    texto = re.sub(r"\s+", " ", texto)
    return texto

# Limpiar columna Country or region
df["country_or_region_clean"] = df["Country or region"].apply(limpiar_texto)

# Clasificar si es región o país
df["tipo_dato"] = np.where(
    df["country_or_region_clean"].isin(regiones_referencia),
    "Region",
    "Country"
)

# Ver resultado
print(df[["Country or region", "country_or_region_clean", "tipo_dato"]])

            Country or region   country_or_region_clean tipo_dato
0                     Finland                   finland   Country
1                      Norway                    norway   Country
2                     Denmark                   denmark   Country
3                     Iceland                   iceland   Country
4                 Switzerland               switzerland   Country
..                        ...                       ...       ...
151                     Yemen                     yemen   Country
152                  Tanzania                  tanzania   Country
153               South Sudan               south sudan   Country
154  Central African Republic  central african republic   Country
155                   Burundi                   burundi   Country

[156 rows x 3 columns]


In [11]:
print(df["tipo_dato"].value_counts())

tipo_dato
Country    156
Name: count, dtype: int64


In [12]:
print(df[df["tipo_dato"] == "Region"][["Country or region"]])

Empty DataFrame
Columns: [Country or region]
Index: []


In [13]:
print(df[df["tipo_dato"] == "Country"][["Country or region"]])

            Country or region
0                     Finland
1                      Norway
2                     Denmark
3                     Iceland
4                 Switzerland
..                        ...
151                     Yemen
152                  Tanzania
153               South Sudan
154  Central African Republic
155                   Burundi

[156 rows x 1 columns]


In [11]:
#ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport (df, title="Raw sale data profiling report", explorative= True)
profile.to_file("raw_sale_data_data_profile2018.html" )

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 412.22it/s]
